# Notebook 2 — Refinamento e Integração dos Dados
**Projeto:** Predição de Evasão Escolar em Pernambuco  
**Disciplina:** Aprendizagem de Máquina | **Entrega:** 11/05/2026
---

## 1. Objetivos

1. Inspecionar os microdados de infraestrutura escolar (Dataset 2)
2. Carregar e limpar as taxas de rendimento por município (`tx_rend_municipios_2024.xlsx` — 5.570 municípios)
3. Agregar os microdados de infraestrutura por município × localização × dependência
4. Construir a variável-alvo (`risco_evasao`) com limiar P75 nacional
5. Integrar os dois datasets e salvar → `dataset_integrado.csv` (usado nos notebooks 3–5)

## 2. Importações

In [45]:
from pathlib import Path
import os

# Garante que o diretório de trabalho é sempre a raiz do projeto
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print(f'Diretório de trabalho: {PROJECT_ROOT}')

Diretório de trabalho: /Users/joaogui/Downloads/1VA_Aprendizado_Maquina


In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

print("Bibliotecas carregadas.")

Bibliotecas carregadas.


## 3. Microdados de Infraestrutura — Inspeção Inicial

In [47]:
# ── Inspecionar sem carregar tudo ─────────────────────────────────────────
df_head = pd.read_csv(
    'data/raw/microdados_ed_basica_2024.csv',
    sep=';', encoding='latin-1', nrows=3
)
print(f"Colunas totais no arquivo: {df_head.shape[1]}")
print(f"Colunas selecionadas para este projeto: 23")
print()
print("Colunas de infraestrutura disponíveis (amostra dos valores):")
cols_infra_raw = [
    'IN_INTERNET','IN_INTERNET_ALUNOS','IN_BANDA_LARGA',
    'IN_BIBLIOTECA','IN_LABORATORIO_INFORMATICA',
    'IN_QUADRA_ESPORTES','IN_AGUA_POTAVEL','IN_ENERGIA_INEXISTENTE',
]
for c in cols_infra_raw:
    print(f"  {c}: {df_head[c].iloc[0]} (0=não tem, 1=tem)")

Colunas totais no arquivo: 426
Colunas selecionadas para este projeto: 23

Colunas de infraestrutura disponíveis (amostra dos valores):
  IN_INTERNET: 1.0 (0=não tem, 1=tem)
  IN_INTERNET_ALUNOS: 0.0 (0=não tem, 1=tem)
  IN_BANDA_LARGA: 0.0 (0=não tem, 1=tem)
  IN_BIBLIOTECA: 0.0 (0=não tem, 1=tem)
  IN_LABORATORIO_INFORMATICA: 0.0 (0=não tem, 1=tem)
  IN_QUADRA_ESPORTES: 0.0 (0=não tem, 1=tem)
  IN_AGUA_POTAVEL: 1.0 (0=não tem, 1=tem)
  IN_ENERGIA_INEXISTENTE: 0.0 (0=não tem, 1=tem)


## 4. Carregamento dos Microdados

In [48]:
# ── Colunas necessárias ───────────────────────────────────────────────────
cols_necessarias = [
    'NO_UF', 'CO_MUNICIPIO', 'SG_UF', 'TP_DEPENDENCIA', 'TP_LOCALIZACAO',
    'IN_INTERNET', 'IN_INTERNET_ALUNOS', 'IN_BANDA_LARGA',
    'IN_BIBLIOTECA', 'IN_BIBLIOTECA_SALA_LEITURA',
    'IN_LABORATORIO_CIENCIAS', 'IN_LABORATORIO_INFORMATICA',
    'IN_QUADRA_ESPORTES', 'IN_ENERGIA_REDE_PUBLICA', 'IN_ENERGIA_INEXISTENTE',
    'IN_AGUA_POTAVEL', 'IN_AGUA_INEXISTENTE', 'IN_ESGOTO_REDE_PUBLICA',
    'IN_ESGOTO_INEXISTENTE', 'IN_BANHEIRO', 'IN_BANHEIRO_PNE',
    'IN_ALIMENTACAO', 'IN_ACESSIBILIDADE_INEXISTENTE',
    'IN_COMPUTADOR', 'QT_SALAS_UTILIZADAS',
]

# Leitura em chunks — necessário para arquivo de 208 MB
# Justificativa: carregar o arquivo inteiro usaria >2 GB de RAM
print("Carregando microdados em chunks de 20.000 linhas...")
chunks = []
for chunk in pd.read_csv(
    'data/raw/microdados_ed_basica_2024.csv',
    sep=';', encoding='latin-1',
    usecols=cols_necessarias,
    chunksize=20000
):
    chunks.append(chunk)

df_micro = pd.concat(chunks, ignore_index=True)
print(f"Total de escolas carregadas: {len(df_micro):,}")
print(f"UFs únicas: {df_micro['NO_UF'].nunique()}")

# Mapear códigos para categorias legíveis
df_micro['unidade_geografica'] = df_micro['NO_UF']
df_micro['dependencia_adm'] = df_micro['TP_DEPENDENCIA'].map(
    {1:'Federal', 2:'Estadual', 3:'Municipal', 4:'Privada'}
)
df_micro['localizacao'] = df_micro['TP_LOCALIZACAO'].map({1:'Urbana', 2:'Rural'})

print(f"\nEscolas por dependência:")
print(df_micro['dependencia_adm'].value_counts().to_string())

Carregando microdados em chunks de 20.000 linhas...
Total de escolas carregadas: 215,545
UFs únicas: 27

Escolas por dependência:
dependencia_adm
Municipal    128999
Privada       52568
Estadual      33254
Federal         724


In [49]:
# ── Mapeamento: coluna bruta → nome padronizado ───────────────────────────
# Usado na agregação por município (Seção 5)
rename_infra = {
    'IN_INTERNET':'pct_internet', 'IN_INTERNET_ALUNOS':'pct_internet_alunos',
    'IN_BANDA_LARGA':'pct_banda_larga', 'IN_BIBLIOTECA':'pct_biblioteca',
    'IN_BIBLIOTECA_SALA_LEITURA':'pct_sala_leitura',
    'IN_LABORATORIO_CIENCIAS':'pct_lab_ciencias',
    'IN_LABORATORIO_INFORMATICA':'pct_lab_informatica',
    'IN_QUADRA_ESPORTES':'pct_quadra', 'IN_ENERGIA_REDE_PUBLICA':'pct_energia_rede',
    'IN_ENERGIA_INEXISTENTE':'pct_sem_energia', 'IN_AGUA_POTAVEL':'pct_agua_potavel',
    'IN_AGUA_INEXISTENTE':'pct_sem_agua', 'IN_ESGOTO_REDE_PUBLICA':'pct_esgoto_rede',
    'IN_ESGOTO_INEXISTENTE':'pct_sem_esgoto', 'IN_BANHEIRO':'pct_banheiro',
    'IN_BANHEIRO_PNE':'pct_banheiro_acessivel', 'IN_ALIMENTACAO':'pct_alimentacao',
    'IN_ACESSIBILIDADE_INEXISTENTE':'pct_sem_acessibilidade',
    'IN_COMPUTADOR':'pct_computador',
}
print(f"Mapeamento de infraestrutura: {len(rename_infra)} colunas")

Mapeamento de infraestrutura: 19 colunas


## 5. Taxas de Rendimento por Município

O arquivo `tx_rend_municipios_2024.xlsx` tem o mesmo formato do Dataset 1, mas na granularidade de **município** em vez de UF — cobrindo os 5.570 municípios do Brasil. Isso nos dará ~50.000 instâncias para modelagem, 125× mais que o pipeline por UF.

In [50]:
# ── Carregar rendimento por município ────────────────────────────────────
print("Carregando taxas de rendimento por município...")
df_rend_mun = pd.read_excel(
    'data/raw/tx_rend_municipios_2024.xlsx',
    sheet_name='MUNICIPIOS ', header=None, skiprows=8
)
df_rend_mun.columns = df_rend_mun.iloc[0].tolist()
df_rend_mun = df_rend_mun.iloc[1:].copy().reset_index(drop=True)
df_rend_mun = df_rend_mun[
    df_rend_mun['NU_ANO_CENSO'].notna() &
    (df_rend_mun['NU_ANO_CENSO'] != 'Fonte: INEP/Censo Escolar da Educação Básica.')
].copy()

# Remover "Pública" (Federal+Estadual+Municipal agregados — derivado, não independente)
df_rend_mun = df_rend_mun[df_rend_mun['NO_DEPENDENCIA'] != 'Pública'].copy()

rename_mun = {
    'SG_UF': 'uf', 'CO_MUNICIPIO': 'cod_municipio', 'NO_MUNICIPIO': 'nome_municipio',
    'NO_CATEGORIA': 'localizacao', 'NO_DEPENDENCIA': 'dependencia_adm',
    '2_CAT_FUN':   'reprov_fund_total',     '2_CAT_FUN_AI': 'reprov_fund_anos_iniciais',
    '2_CAT_FUN_AF':'reprov_fund_anos_finais','2_CAT_MED':    'reprov_med_total',
    '2_CAT_MED_01':'reprov_med_1serie',     '2_CAT_MED_02': 'reprov_med_2serie',
    '2_CAT_MED_03':'reprov_med_3serie',
    '3_CAT_FUN':   'abandono_fund_total',   '3_CAT_FUN_AI': 'abandono_fund_anos_iniciais',
    '3_CAT_FUN_AF':'abandono_fund_anos_finais','3_CAT_MED':  'abandono_med_total',
    '3_CAT_MED_01':'abandono_med_1serie',   '3_CAT_MED_02': 'abandono_med_2serie',
    '3_CAT_MED_03':'abandono_med_3serie',
}
df_rend_mun = df_rend_mun.rename(columns=rename_mun)

num_cols_mun = [v for v in rename_mun.values() if v not in ('uf','cod_municipio','nome_municipio','localizacao','dependencia_adm')]
for col in num_cols_mun:
    if col in df_rend_mun.columns:
        df_rend_mun[col] = pd.to_numeric(df_rend_mun[col], errors='coerce')

df_rend_mun['abandono_geral'] = df_rend_mun[['abandono_fund_total','abandono_med_total']].mean(axis=1)
df_rend_mun['cod_municipio']  = df_rend_mun['cod_municipio'].astype(str).str.strip()

print(f"Rendimento municipal — shape: {df_rend_mun.shape}")
print(f"Municípios únicos: {df_rend_mun['cod_municipio'].nunique()}")
print(f"Localizações: {df_rend_mun['localizacao'].unique().tolist()}")
print(f"Dependências: {df_rend_mun['dependencia_adm'].unique().tolist()}")
print(f"Municípios de PE: {df_rend_mun[df_rend_mun['uf']=='PE']['cod_municipio'].nunique()}")

Carregando taxas de rendimento por município...
Rendimento municipal — shape: (50197, 62)
Municípios únicos: 5570
Localizações: ['Total', 'Urbana', 'Rural']
Dependências: ['Total', 'Estadual', 'Municipal', 'Federal', 'Privada']
Municípios de PE: 185


## 6. Agregação dos Microdados por Município

Usa `df_micro` (carregado na Seção 4, com `CO_MUNICIPIO` e `SG_UF`), gerando todas as combinações de localização × dependência por município — compatível com o Dataset de Rendimento.

In [51]:
df_micro['CO_MUNICIPIO'] = df_micro['CO_MUNICIPIO'].astype(str).str.strip()

DEP_MAP = {1: 'Federal', 2: 'Estadual', 3: 'Municipal', 4: 'Privada'}
LOC_MAP = {1: 'Urbana',  2: 'Rural'}

def agg_infra_mun(subset):
    result = {v: subset[k].mean() * 100
              for k, v in rename_infra.items() if k in subset.columns}
    result['qt_salas_media'] = subset['QT_SALAS_UTILIZADAS'].mean()
    result['n_escolas'] = len(subset)
    return pd.Series(result)

# Gerar as 15 combinações: Total×Total, Loc×Total, Total×Dep, Loc×Dep
print("Agregando microdados por município × localização × dependência...")
combos = [('Total', None, 'Total', None)]
for lc, ln in LOC_MAP.items():
    combos.append((ln, lc, 'Total', None))
for dc, dn in DEP_MAP.items():
    combos.append(('Total', None, dn, dc))
for lc, ln in LOC_MAP.items():
    for dc, dn in DEP_MAP.items():
        combos.append((ln, lc, dn, dc))

parts_mun = []
for loc_name, loc_code, dep_name, dep_code in combos:
    mask = pd.Series([True] * len(df_micro), index=df_micro.index)
    if loc_code:
        mask &= df_micro['TP_LOCALIZACAO'] == loc_code
    if dep_code:
        mask &= df_micro['TP_DEPENDENCIA'] == dep_code
    sub = df_micro[mask]
    if len(sub) == 0:
        continue
    agg = sub.groupby('CO_MUNICIPIO').apply(
        agg_infra_mun, include_groups=False
    ).reset_index()
    agg['localizacao']    = loc_name
    agg['dependencia_adm'] = dep_name
    parts_mun.append(agg)

df_infra_mun = pd.concat(parts_mun, ignore_index=True)
print(f"Infraestrutura municipal: {df_infra_mun.shape}")
print(f"Municípios cobertos: {df_infra_mun['CO_MUNICIPIO'].nunique()}")

Agregando microdados por município × localização × dependência...
Infraestrutura municipal: (53497, 24)
Municípios cobertos: 5570


## 7. Integração e Dataset Final

In [52]:
# ── Merge por cod_municipio × localizacao × dependencia_adm ──────────────
df_mun_final = df_rend_mun.merge(
    df_infra_mun,
    left_on=['cod_municipio', 'localizacao', 'dependencia_adm'],
    right_on=['CO_MUNICIPIO',  'localizacao', 'dependencia_adm'],
    how='left'
).drop(columns='CO_MUNICIPIO')

# ── Variável-alvo: P75 nacional do abandono_geral ─────────────────────────
limiar_mun = df_mun_final['abandono_geral'].quantile(0.75)
df_mun_final['risco_evasao'] = (df_mun_final['abandono_geral'] > limiar_mun).astype(int)
df_mun_final = df_mun_final.dropna(subset=['abandono_geral']).reset_index(drop=True)

print(f"Dataset municipal — shape: {df_mun_final.shape}")
print(f"Municípios: {df_mun_final['cod_municipio'].nunique()} | UFs: {df_mun_final['uf'].nunique()}")
print(f"Limiar P75 abandono_geral: {limiar_mun:.2f}%")
print(f"Classe 0 (baixo risco): {(df_mun_final['risco_evasao']==0).sum()}")
print(f"Classe 1 (alto risco):  {(df_mun_final['risco_evasao']==1).sum()} "
      f"({df_mun_final['risco_evasao'].mean()*100:.1f}%)")

infra_cobertura = df_mun_final['pct_internet'].notna().mean() * 100
print(f"\nCobertura de infraestrutura (pct_internet não-nulo): {infra_cobertura:.1f}%")

print(f"\nMunicípios de PE (amostra):")
pe_mun = df_mun_final[
    (df_mun_final['uf']=='PE') &
    (df_mun_final['localizacao']=='Total') &
    (df_mun_final['dependencia_adm']=='Total')
][['nome_municipio','abandono_geral','risco_evasao']].sort_values('abandono_geral', ascending=False)
print(pe_mun.head(10).to_string(index=False))

# ── Salvar ─────────────────────────────────────────────────────────────────
df_mun_final.to_csv('data/processed/dataset_integrado.csv', index=False)
print(f"\nSalvo: data/processed/dataset_integrado.csv  ← dataset principal da modelagem")
print(f"Instâncias: {len(df_mun_final)} ({len(df_mun_final)//398}× mais que a versão por UF)")
print(f"Municípios: {df_mun_final['cod_municipio'].nunique()} | UFs: {df_mun_final['uf'].nunique()}")

Dataset municipal — shape: (50140, 84)
Municípios: 5570 | UFs: 27
Limiar P75 abandono_geral: 2.05%
Classe 0 (baixo risco): 37715
Classe 1 (alto risco):  12425 (24.8%)

Cobertura de infraestrutura (pct_internet não-nulo): 100.0%

Municípios de PE (amostra):
      nome_municipio  abandono_geral  risco_evasao
              Brejão            3.00             1
      Santa Filomena            2.80             1
          Santa Cruz            2.75             1
                 Exu            2.70             1
         Itaquitinga            2.55             1
            Floresta            2.50             1
Carnaubeira da Penha            2.15             1
          Paranatama            2.10             1
            Itapetim            2.00             0
           Agrestina            1.90             0

Salvo: data/processed/dataset_integrado.csv  ← dataset principal da modelagem
Instâncias: 50140 (125× mais que a versão por UF)
Municípios: 5570 | UFs: 27
